<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/10_exgaussian_default_priors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 10 — Ex-Gaussian with default priors

Fit the same distributional ex-Gaussian structure using Bambi defaults. This parallels the chapter’s comparison between carefully chosen priors and software defaults.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 10.1 Software-default priors

What prior assumptions does Bambi supply for this ex-Gaussian model when we do not specify them ourselves?

The formula is unchanged. We deliberately omit `priors=` so the notebook shows exactly what Bambi supplies by default. `center_predictors=False` remains explicit because baseline at `Days = 0` is scientifically meaningful.

In [ ]:
formula = bmb.Formula(
    "Reaction ~ Days + (1 + Days | Subject)",
    "sigma ~ 1 + (1 | Subject)",
    "nu ~ 1",
)

model = bmb.Model(
    formula, sleep, family="exgaussian", categorical="Subject",
    center_predictors=False,
)
model

In [ ]:
prior = model.prior_predictive(draws=500, random_seed=RANDOM_SEED)
azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 10.2 Posterior changes under defaults

Which posterior features change when the custom ex-Gaussian priors are replaced by Bambi’s defaults?

In [ ]:
idata = model.fit(draws=1000, tune=2000, chains=4, target_accept=0.95, random_seed=RANDOM_SEED)
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))


In [ ]:
azs.summary(idata, ci_prob=0.90, ci_kind="hdi", round_to=2)

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional="Days",
    average_by="Subject",
    target="Reaction",
    prob=[0.50, 0.90],
)

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional=["Days", "Subject"],
    target="Reaction",
    prob=0.90,
    subplot_kwargs={"main": "Days", "panel": "Subject"},
    fig_kwargs={"wrap": 6},
)

# 10.3 Predictive robustness to defaults

Do Bambi’s default priors materially change the model’s predictive behavior?

In [ ]:
model.predict(
    idata,
    kind="response",
    inplace=True,
    random_seed=RANDOM_SEED,
)

azp.plot_ppc_dist(
    idata,
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 10.4 Parameter-level sensitivity

Which parameters appear prior-sensitive even when the overall predictions remain similar?